# Fine-tune Mô Hình Ngôn Ngữ Bằng QLoRA Trên Google Colab

## Mục tiêu
Huấn luyện thêm (fine-tune) một mô hình ngôn ngữ theo dữ liệu hội thoại tùy chỉnh bằng kỹ thuật **QLoRA 4-bit** để tiết kiệm VRAM, sau đó lưu adapter để tái sử dụng.

## Kết quả mong đợi
- Có adapter LoRA đã huấn luyện.
- Có thể gắn adapter vào base model để suy luận.
- Có cell chat test để đánh giá chất lượng phản hồi.

## Công nghệ sử dụng
- Python, PyTorch
- Hugging Face Transformers, Datasets, PEFT, TRL
- BitsAndBytes (quantization 4-bit)
- Google Colab + Google Drive


## Bước 1: Kết Nối Google Drive

Kết nối Google Drive để:
- Đọc model/dataset đã lưu sẵn.
- Lưu adapter sau khi fine-tune để không mất dữ liệu khi Colab reset runtime.


In [6]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Bước 2: Chuẩn Bị Thư Mục Lưu Trữ

Tạo thư mục trong Drive để quản lý:
- Base model
- Adapter LoRA
- Các checkpoint hoặc output trung gian

Việc tổ chức thư mục rõ ràng giúp tái lập thí nghiệm và trình bày dự án tốt hơn trong CV.


In [5]:
!mkdir -p /content/drive/MyDrive/AI_models

## Bước 3: Cài Đặt Thư Viện

Cài các gói cần thiết cho pipeline huấn luyện:
- `torch`, `torchvision`, `torchaudio`: backend tính toán GPU
- `transformers`: load tokenizer/model
- `accelerate`: tối ưu chạy nhiều thiết bị
- `bitsandbytes`: lượng tử hóa 4-bit
- `sentencepiece`: tokenizer cho nhiều dòng model
- `trl`: SFTTrainer phục vụ supervised fine-tuning


In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.46.3 accelerate==1.2.1 bitsandbytes==0.45.0 "sentencepiece>=0.2.0"

## Bước 4: Tải Dataset Lên Colab

Upload file dữ liệu định dạng `jsonl` để huấn luyện.

### Yêu cầu dữ liệu
- Mỗi dòng là 1 mẫu huấn luyện hợp lệ.
- Nội dung nên nhất quán về phong cách, domain và ngôn ngữ.
- Dữ liệu sạch giúp giảm lỗi sinh nội dung và tăng chất lượng hội thoại.


In [ ]:
from google.colab import files
uploaded = files.upload()

# Lấy tên file vừa upload
data_path = list(uploaded.keys())[0]
print("Dataset uploaded:", data_path)

## Bước 5: Fine-tune Bằng QLoRA Và Lưu Adapter

Quy trình gồm:
1. Load dataset từ file `jsonl`.
2. Load tokenizer và thiết lập `pad_token`.
3. Load base model ở chế độ 4-bit (NF4) để tiết kiệm bộ nhớ.
4. Chuẩn bị mô hình cho k-bit training.
5. Gắn LoRA vào các projection layers trọng yếu.
6. Cấu hình SFTTrainer và bắt đầu huấn luyện.
7. Lưu `LoRA adapter` + tokenizer vào Google Drive.

### Vì sao dùng QLoRA?
- Giảm chi phí phần cứng đáng kể.
- Vẫn giữ chất lượng fine-tune tốt cho nhiều bài toán hội thoại.
- Phù hợp môi trường Colab GPU phổ thông.


In [ ]:
!pip install -U trl

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# =========================
# 1. ĐƯỜNG DẪN
# =========================
model_path = "/content/drive/MyDrive/AI_models/models/snapshots/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2"
data_path = "/content/train.jsonl"   # <- sửa link dataset ở đây
save_dir = "/content/drive/MyDrive/qlora_adapter"
os.makedirs(save_dir, exist_ok=True)

# =========================
# 2. LOAD DATASET
# =========================
dataset = load_dataset("json", data_files=data_path, split="train")
print(dataset)
print(dataset[0])

# =========================
# 3. TOKENIZER
# =========================
tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =========================
# 4. LOAD MODEL 4BIT
# =========================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,

model.config.use_cache = False
model.gradient_checkpointing_enable()

# =========================
# 5. CHUẨN BỊ CHO QLORA
# =========================
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# =========================
# 6. TRAIN CONFIG
# =========================
training_args = SFTConfig(
    output_dir="/content/qlora_temp_output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=1,
    save_steps=50,
    save_total_limit=2,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    warmup_steps=5,
    lr_scheduler_type="cosine",
    report_to="none",
    max_grad_norm=0.0
)

# =========================
# 7. TRAIN
# =========================
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

trainer.train()

# =========================
# 8. LƯU ADAPTER LORA VÀ TOKENIZER VÀO DRIVE
# =========================
trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print("Đã lưu LoRA adapter vào:", save_dir)

## Bước 6: Nạp Adapter Đã Huấn Luyện Để Suy Luận

Sau khi train:
- Load lại base model (4-bit).
- Gắn adapter LoRA từ Drive.
- Tạo prompt hội thoại theo `chat template`.
- Sinh phản hồi để kiểm tra chất lượng mô hình sau fine-tune.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TextStreamer
from peft import PeftModel

base_model_path = "/content/drive/MyDrive/AI_models/models/snapshots/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2"
adapter_path = "/content/drive/MyDrive/qlora_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_path, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

messages = [
    {"role": "system", "content": "Bạn là một trợ lý AI thân thiện."},
    {"role": "user", "content": "Hãy tự giới thiệu ngắn gọn."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(model.device)

_ = model.generate(
    **inputs,
    max_new_tokens=200,
    streamer=streamer,
    pad_token_id=tokenizer.eos_token_id
)

## Bước 7: Chat Vòng Lặp Để Đánh Giá Thực Tế

Cell này chạy chế độ hội thoại liên tục:
- Người dùng nhập câu hỏi nhiều lượt.
- Mô hình phản hồi theo ngữ cảnh gần nhất.
- Dùng để đánh giá độ ổn định, phong cách trả lời và độ phù hợp domain.

Gõ `exit` để kết thúc phiên chat.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TextStreamer
from transformers.utils import logging

logging.set_verbosity_error()

model_path = "/content/drive/MyDrive/AI_models/models/snapshots/8afb486c1db24fe5011ec46dfbe5b5dccdb575c2"

# Quantization 4bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto"
)

# streamer
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

messages = [
    {"role": "system", "content": "Bạn là một trợ lý AI thân thiện."}
]

# Giữ system message và 5 cuộc hôi thoại gần nhất
if len(messages) > 11:
        messages = [messages[0]] + messages[-10:]

while True:

    user_input = input("You: ")

    if user_input.lower() == "exit":
        break

    messages.append({"role": "user", "content": user_input})

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    print("AI: ", end="", flush=True)

    outputs = model.generate(
        **inputs,
        max_new_tokens=600,
        max_length=None,
        streamer=streamer,
        pad_token_id=tokenizer.eos_token_id
    )

    print()

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    messages.append({"role": "assistant", "content": response})